<a href="https://colab.research.google.com/github/SampMark/ETL-de-dados-da-PNP/blob/main/geocodifica%C3%A7%C3%A3o_RFEPT/Geocoding_from_Web_Scraping_and_merge_RFEPT_map_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pandas geopy requests

In [4]:
# -*- coding: utf-8 -*-

"""
Geocodificação das unidades da Rede Federal a partir de endereço, CEP,
município e UF.

Entrada:
- CSV público do GitHub com colunas:
  UID, Nome_IF, Campus_IF, Tipo_Unidade, UF, Municipio,
  Endereco_Padronizado, CEP, etc.

Saída:
- CSV com latitude, longitude, status, precisão e consulta usada.
"""

import json
import re
import time
from pathlib import Path

import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

import gspread
from google.colab import auth
from google.auth import default
from gspread_dataframe import set_with_dataframe
print("Setup concluído!")

Setup concluído!


In [2]:
# ---------------------------------------------------------------------
# 1. Configurações
# ---------------------------------------------------------------------

URL = (
    "https://raw.githubusercontent.com/SampMark/files/refs/heads/main/"
    "Web-Scraping-and-merge-RFEPT-map-public%20-%20Web-Scraping-and-merge-RFEPT-map.csv"
)

ARQUIVO_SAIDA = "rfept_unidades_geocodificadas.csv"
ARQUIVO_CACHE = "cache_geocodificacao_rfept.json"

# Utilizar um identificador real da sua aplicação.
# Não usar user-agent genérico.
USER_AGENT = "rfept-geocoder-marcus-sampaio/1.0 contato:prof.marcus.sampaio@gmail.com"

# Nominatim público: manter 1 req/s ou mais lento.
MIN_DELAY_SECONDS = 1.2

# Timeout por requisição
TIMEOUT = 15


# ---------------------------------------------------------------------
# 2. Funções auxiliares
# ---------------------------------------------------------------------

def limpar_texto(valor):
    """Limpa espaços, quebras e alguns ruídos comuns."""
    if pd.isna(valor):
        return ""

    texto = str(valor).strip()

    # Normaliza espaços
    texto = re.sub(r"\s+", " ", texto)

    # Corrige alguns artefatos comuns observados em bases raspadas
    texto = texto.replace("–", "-")
    texto = texto.replace("—", "-")
    texto = texto.replace("º", "")
    texto = texto.replace("nº", "n")
    texto = texto.replace("Nº", "n")
    texto = texto.replace("s/nº", "s/n")
    texto = texto.replace("s/no", "s/n")

    # Corrige caso pontual observado no CSV
    texto = texto.replace("Antÿnio", "Antônio")

    # Remove "CEP:" do endereço quando aparece dentro do campo
    texto = re.sub(r"\bCEP\s*:?", "", texto, flags=re.IGNORECASE)

    # Corrige CEP quebrado como "37890, 000"
    texto = re.sub(r"(\d{5})\s*,\s*(\d{3})", r"\1-\2", texto)

    # Remove espaços antes de vírgula
    texto = re.sub(r"\s+,", ",", texto)

    return texto.strip()


def limpar_cep(valor):
    """Mantém CEP no padrão 00000-000 quando possível."""
    if pd.isna(valor):
        return ""

    digitos = re.sub(r"\D", "", str(valor))

    if len(digitos) == 8:
        return f"{digitos[:5]}-{digitos[5:]}"

    return ""


def montar_consultas(row):
    """
    Gera consultas em camadas, da mais precisa para a menos precisa.
    Cada item retorna: (consulta, nivel_precisao).
    """

    endereco = limpar_texto(row.get("Endereco_Padronizado", ""))
    cep = limpar_cep(row.get("CEP", ""))
    municipio = limpar_texto(row.get("Municipio", ""))
    uf = limpar_texto(row.get("UF", ""))
    campus = limpar_texto(row.get("Campus_IF", ""))
    nome_if = limpar_texto(row.get("Nome_IF", ""))

    consultas = []

    # 1. Endereço completo já padronizado
    if endereco:
        consultas.append((
            f"{endereco}, Brasil",
            "endereco_padronizado"
        ))

    # 2. Endereço + CEP
    if endereco and cep:
        consultas.append((
            f"{endereco}, {cep}, Brasil",
            "endereco_cep"
        ))

    # 3. CEP + município + UF
    if cep and municipio and uf:
        consultas.append((
            f"{cep}, {municipio}, {uf}, Brasil",
            "cep_municipio_uf"
        ))

    # 4. Nome institucional + município
    if campus and nome_if and municipio and uf:
        consultas.append((
            f"{campus}, {nome_if}, {municipio}, {uf}, Brasil",
            "nome_unidade"
        ))

    # 5. Município/UF — baixa precisão
    if municipio and uf:
        consultas.append((
            f"{municipio}, {uf}, Brasil",
            "municipio_uf"
        ))

    # Remove duplicatas preservando ordem
    vistas = set()
    unicas = []
    for consulta, precisao in consultas:
        chave = consulta.lower()
        if chave not in vistas:
            vistas.add(chave)
            unicas.append((consulta, precisao))

    return unicas


def carregar_cache(caminho):
    caminho = Path(caminho)
    if caminho.exists():
        return json.loads(caminho.read_text(encoding="utf-8"))
    return {}


def salvar_cache(cache, caminho):
    Path(caminho).write_text(
        json.dumps(cache, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )


def geocodificar_com_cache(consulta, geocode, cache):
    """
    Consulta o geocoder usando cache local.
    Retorna dicionário padronizado.
    """

    if consulta in cache:
        return cache[consulta]

    try:
        location = geocode(consulta)

        if location is None:
            resultado = {
                "status": "nao_encontrado",
                "latitude": None,
                "longitude": None,
                "endereco_retornado": None,
                "osm_type": None,
                "osm_id": None,
                "importance": None,
                "raw": None,
            }
        else:
            raw = location.raw or {}
            resultado = {
                "status": "ok",
                "latitude": location.latitude,
                "longitude": location.longitude,
                "endereco_retornado": location.address,
                "osm_type": raw.get("osm_type"),
                "osm_id": raw.get("osm_id"),
                "importance": raw.get("importance"),
                "raw": raw,
            }

    except (GeocoderTimedOut, GeocoderServiceError) as exc:
        resultado = {
            "status": f"erro_servico: {type(exc).__name__}",
            "latitude": None,
            "longitude": None,
            "endereco_retornado": None,
            "osm_type": None,
            "osm_id": None,
            "importance": None,
            "raw": None,
        }

    except Exception as exc:
        resultado = {
            "status": f"erro: {type(exc).__name__}",
            "latitude": None,
            "longitude": None,
            "endereco_retornado": None,
            "osm_type": None,
            "osm_id": None,
            "importance": None,
            "raw": None,
        }

    cache[consulta] = resultado
    return resultado


def escolher_geocodificacao(row, geocode, cache):
    """
    Tenta várias consultas até encontrar coordenadas.
    """

    consultas = montar_consultas(row)

    for consulta, precisao in consultas:
        resultado = geocodificar_com_cache(consulta, geocode, cache)

        if resultado["status"] == "ok":
            return {
                "geocode_status": "ok",
                "geocode_precisao": precisao,
                "geocode_query": consulta,
                "latitude": resultado["latitude"],
                "longitude": resultado["longitude"],
                "geocode_endereco_retornado": resultado["endereco_retornado"],
                "geocode_osm_type": resultado["osm_type"],
                "geocode_osm_id": resultado["osm_id"],
                "geocode_importance": resultado["importance"],
            }

    return {
        "geocode_status": "falha",
        "geocode_precisao": None,
        "geocode_query": None,
        "latitude": None,
        "longitude": None,
        "geocode_endereco_retornado": None,
        "geocode_osm_type": None,
        "geocode_osm_id": None,
        "geocode_importance": None,
    }


# ---------------------------------------------------------------------
# 3. Carregar e preparar dados
# ---------------------------------------------------------------------

df = pd.read_csv(URL, dtype=str, keep_default_na=False)

# Remove espaços nos nomes de colunas
df.columns = [c.strip() for c in df.columns]

colunas_obrigatorias = {
    "UID",
    "Sigla_IF",
    "Nome_IF",
    "Campus_IF",
    "Tipo_Unidade",
    "UF",
    "Municipio",
    "Endereco_Padronizado",
    "CEP",
}

faltantes = colunas_obrigatorias - set(df.columns)
if faltantes:
    raise ValueError(f"Colunas obrigatórias ausentes no CSV: {faltantes}")

# Limpeza básica
for coluna in ["Endereco_Padronizado", "CEP", "Municipio", "UF", "Campus_IF", "Nome_IF"]:
    df[coluna] = df[coluna].apply(limpar_texto)

df["CEP"] = df["CEP"].apply(limpar_cep)

# Campo auxiliar: endereço final candidato
df["Endereco_Geocodificacao_Base"] = df.apply(
    lambda row: montar_consultas(row)[0][0] if montar_consultas(row) else "",
    axis=1
)


# ---------------------------------------------------------------------
# 4. Geocodificar
# ---------------------------------------------------------------------

cache = carregar_cache(ARQUIVO_CACHE)

geolocator = Nominatim(
    user_agent=USER_AGENT,
    timeout=TIMEOUT,
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=MIN_DELAY_SECONDS,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=False,
)

resultados = []

total = len(df)

for i, row in df.iterrows():
    print(f"[{i + 1}/{total}] Geocodificando: {row['UID']}")

    resultado = escolher_geocodificacao(row, geocode, cache)
    resultados.append(resultado)

    # Salva cache progressivamente para permitir retomada se interromper
    if (i + 1) % 10 == 0:
        salvar_cache(cache, ARQUIVO_CACHE)

salvar_cache(cache, ARQUIVO_CACHE)

geo = pd.DataFrame(resultados)

df_geo = pd.concat([df, geo], axis=1)


# ---------------------------------------------------------------------
# 5. Classificar qualidade do resultado
# ---------------------------------------------------------------------

def classificar_confianca(precisao):
    if precisao in {"endereco_padronizado", "endereco_cep"}:
        return "alta"
    if precisao in {"cep_municipio_uf", "nome_unidade"}:
        return "media"
    if precisao == "municipio_uf":
        return "baixa"
    return "sem_resultado"


df_geo["geocode_confianca"] = df_geo["geocode_precisao"].apply(classificar_confianca)

# Marcação explícita para revisar manualmente
df_geo["revisar_manual"] = df_geo["geocode_confianca"].isin(
    ["baixa", "sem_resultado"]
)


# ---------------------------------------------------------------------
# 6. Exportar
# ---------------------------------------------------------------------

colunas_saida = [
    "UID",
    "Ano_Criacao",
    "Sigla_IF",
    "Nome_IF",
    "Campus_IF",
    "Tipo_Unidade",
    "Regioes",
    "UF",
    "Municipio",
    "Endereco_Padronizado",
    "CEP",
    "Telefone",
    "E-mail",
    "Site",
    "Dirigente",
    "Fonte",
    "Endereco_Geocodificacao_Base",
    "latitude",
    "longitude",
    "geocode_status",
    "geocode_precisao",
    "geocode_confianca",
    "geocode_query",
    "geocode_endereco_retornado",
    "geocode_osm_type",
    "geocode_osm_id",
    "geocode_importance",
    "revisar_manual",
]

colunas_saida = [c for c in colunas_saida if c in df_geo.columns]

df_geo[colunas_saida].to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8-sig"
)

print("\nConcluído.")
print(f"Arquivo gerado: {ARQUIVO_SAIDA}")
print(f"Cache salvo em: {ARQUIVO_CACHE}")

print("\nResumo da geocodificação:")
print(df_geo["geocode_confianca"].value_counts(dropna=False))

print("\nRegistros para revisão manual:")
print(
    df_geo.loc[
        df_geo["revisar_manual"],
        ["UID", "Sigla_IF", "Campus_IF", "Municipio", "UF", "Endereco_Padronizado", "CEP", "geocode_query"]
    ]
)

[1/734] Geocodificando: ifsuldeminas_reitoria_do_instituto_federal_do_sul_de_minas_gerais
[2/734] Geocodificando: ifsuldeminas_campus_avancado_carmo_de_minas
[3/734] Geocodificando: ifsuldeminas_campus_avancado_tres_coracoes
[4/734] Geocodificando: ifsuldeminas_campus_inconfidentes
[5/734] Geocodificando: ifsuldeminas_campus_machado
[6/734] Geocodificando: ifsuldeminas_campus_muzambinho
[7/734] Geocodificando: ifsuldeminas_campus_passos
[8/734] Geocodificando: ifsuldeminas_campus_pocos_de_caldas
[9/734] Geocodificando: ifsuldeminas_campus_pouso_alegre
[10/734] Geocodificando: cefet_mg_reitoria_do_centro_federal_de_educacao_tecnologica_de_minas_gerais
[11/734] Geocodificando: cefet_mg_uned_araxa
[12/734] Geocodificando: cefet_mg_uned_contagem
[13/734] Geocodificando: cefet_mg_uned_curvelo
[14/734] Geocodificando: cefet_mg_uned_divinopolis
[15/734] Geocodificando: cefet_mg_uned_leopoldina
[16/734] Geocodificando: cefet_mg_uned_nepomuceno
[17/734] Geocodificando: cefet_mg_uned_timoteo
[18

In [3]:
# Autenticar no Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Spreadsheet ID
spreadsheet_id = '1CkV8Y7Pesw2BqpWaTpCaRLQZ1Eq-gYcWeEdowfRga1g'

try:
    # Abrir a planilha na primeira guia
    sh = gc.open_by_key(spreadsheet_id)
    worksheet = sh.get_worksheet(0)

    # Exportar 'df_geo'
    set_with_dataframe(worksheet, df_geo)

    # Exibir link de acesso público
    url = f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}"
    print(f"Successfully exported 'df_geo' to spreadsheet: {sh.title}")
    print(f"Link to spreadsheet: {url}")
except Exception as e:
    print(f"Error exporting to Google Sheets: {e}")

Successfully exported 'df_geo' to spreadsheet: Geocoding_from_Web-Scraping-and-merge-RFEPT-map_advanced
Link to spreadsheet: https://docs.google.com/spreadsheets/d/1CkV8Y7Pesw2BqpWaTpCaRLQZ1Eq-gYcWeEdowfRga1g
